# Step 3: XGBoost Training — Demo Notebook
---
**Goal**: Train XGBoost, analyze residuals, and validate model performance.

### Contents:
1. Load feature matrices
2. Training curve visualization
3. Feature importance plot
4. Residual analysis (by store type, family, date)
5. Prediction vs actual scatter
6. Hyperparameter tuning results

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('tab10')
plt.rcParams['figure.dpi'] = 100

sys.path.insert(0, os.path.abspath(''))
from config import *
from utils import Timer, rmsle

print('Libraries loaded.')

In [ ]:
# Load feature matrices
X_train = pd.read_csv(os.path.join(DATA_FEATURES_DIR, 'X_train.csv'))
X_valid = pd.read_csv(os.path.join(DATA_FEATURES_DIR, 'X_valid.csv'))

y_train = X_train['sales'].copy()
y_valid = X_valid['sales'].copy()
date_valid = X_valid['date'].copy()

X_train.drop(columns=['sales', 'date', 'id'], inplace=True, errors='ignore')
X_valid.drop(columns=['sales', 'date', 'id'], inplace=True, errors='ignore')

common_cols = X_train.columns.intersection(X_valid.columns).tolist()
X_train = X_train[common_cols]
X_valid = X_valid[common_cols]

print(f'X_train: {X_train.shape[0]:,} rows x {X_train.shape[1]} cols')
print(f'X_valid: {X_valid.shape[0]:,} rows x {X_valid.shape[1]} cols')
print(f'y_train mean: {y_train.mean():.2f}, y_valid mean: {y_valid.mean():.2f}')

## 1. Training Curves

XGBoost with early stopping after 30 rounds of no improvement.

In [ ]:
# Train XGBoost with logging
params = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse',
    'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'min_child_weight': 5,
    'reg_alpha': 1.0, 'reg_lambda': 1.0,
    'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
}

evals_result = {}
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

with Timer('Training'):
    model = xgb.train(
        params, dtrain, num_boost_round=500,
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        early_stopping_rounds=30, evals_result=evals_result, verbose_eval=50,
    )

# Plot training curves
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(evals_result['train']['rmse'], label='Train RMSE', color='#3498DB', linewidth=1, alpha=0.7)
ax.plot(evals_result['valid']['rmse'], label='Valid RMSE', color='#E74C3C', linewidth=1.5)
ax.axvline(model.best_iteration, color='green', linestyle='--', alpha=0.7, label=f'Best={model.best_iteration}')
ax.set_title('XGBoost Training Curves', fontsize=13, fontweight='bold')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('RMSE')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Best iteration: {model.best_iteration}')
print(f'Best valid RMSE: {model.best_score:.2f}')

In [ ]:
y_pred = model.predict(dvalid)
val_rmsle = rmsle(y_valid, y_pred)
print(f'Validation RMSLE: {val_rmsle:.4f}')

## 2. Feature Importance (Top 20 by Gain)

In [ ]:
importance = model.get_score(importance_type='gain')
imp_df = pd.DataFrame({'feature': list(importance.keys()), 'gain': list(importance.values())})
imp_df = imp_df.sort_values('gain', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette('rocket_r', len(imp_df))
ax.barh(range(len(imp_df)), imp_df['gain'].values[::-1] / 1e9, color=colors)
ax.set_yticks(range(len(imp_df)))
ax.set_yticklabels(imp_df['feature'].values[::-1])
ax.set_title('Top 20 Feature Importance (Gain, Billions)', fontsize=13, fontweight='bold')
ax.set_xlabel('Gain (Billions)')
plt.tight_layout()
plt.show()

## 3. Residual Analysis

Where does the model make the largest errors?

In [ ]:
residuals = y_valid - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residual distribution
axes[0].hist(residuals, bins=100, color='#3498DB', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title(f'Residual Distribution (mean={residuals.mean():.1f}, std={residuals.std():.1f})',
                   fontsize=11, fontweight='bold')
axes[0].set_xlabel('Residual (y_true - y_pred)')
axes[0].set_ylabel('Count')

# Predicted vs Actual
max_val = max(y_valid.max(), y_pred.max())
axes[1].scatter(y_valid, y_pred, s=1, alpha=0.3, color='#2C3E50')
axes[1].plot([0, max_val], [0, max_val], color='red', linestyle='--', linewidth=1, label='Perfect')
axes[1].set_xlim(0, max_val * 0.3)
axes[1].set_ylim(0, max_val * 0.3)
axes[1].set_title('Predicted vs Actual (trimmed)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Actual Sales')
axes[1].set_ylabel('Predicted Sales')
axes[1].legend()

# Residuals over time
res_by_date = pd.DataFrame({'date': pd.to_datetime(date_valid), 'residual': residuals})
daily_res = res_by_date.groupby('date')['residual'].mean()
axes[2].plot(daily_res.index, daily_res.values, marker='o', markersize=3, linewidth=1, color='#E67E22')
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.5)
axes[2].set_title('Mean Residual by Date', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Mean Residual')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. RMSLE by Product Family

Which products are hardest to predict?

In [ ]:
# Find family columns from one-hot encoding
family_cols = [c for c in common_cols if c.startswith('family_')]
families = X_valid[family_cols].idxmax(axis=1).str.replace('family_', '')

family_scores = {}
for fam in families.unique():
    mask = families == fam
    family_scores[fam] = rmsle(y_valid[mask], y_pred[mask])

sorted_fams = sorted(family_scores.items(), key=lambda x: x[1])

fig, ax = plt.subplots(figsize=(10, 8))
names = [f[0] for f in sorted_fams]
scores = [f[1] for f in sorted_fams]
colors = ['#27AE60' if s < np.median(scores) else '#E74C3C' for s in scores]
ax.barh(names, scores, color=colors, edgecolor='white')
ax.set_title('RMSLE by Product Family', fontsize=13, fontweight='bold')
ax.set_xlabel('RMSLE')
ax.axvline(val_rmsle, color='black', linestyle='--', linewidth=1, alpha=0.5, label=f'Overall={val_rmsle:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

- **Baseline RMSLE: ~0.236** — strong starting point
- **Top features**: rolling_mean_7d, rolling_mean_14d, lag_7 (all time-series features)
- **Residuals** are roughly symmetric (mean≈0), well-calibrated
- **Hardest families**: Lingerie, Hardware, School Supplies (high variance, low frequency)
- **Easiest families**: Dairy, Bread/Bakery, Grocery I (high volume, stable patterns)

→ Proceed to Step 4 for submission generation.